# Bonus Analytics - B3 Monte Carlo, B4 Efficient Frontier, B5 Email Report
**Bluestock Fintech Capstone**

## B3 - Monte Carlo NAV Simulation
Projects NAV over 5 years with 1,000 simulated paths showing expected path and uncertainty bands.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')

ROOT    = Path().resolve().parent
PROC    = ROOT / 'data' / 'processed'
REPORTS = ROOT / 'reports'

nav_df  = pd.read_csv(PROC / '02_nav_history_clean.csv')
nav_df['date'] = pd.to_datetime(nav_df['date'])
fund_df = pd.read_csv(PROC / '01_fund_master_clean.csv')
fund_names = fund_df.set_index('amfi_code')['scheme_name'].to_dict()
nav_df = nav_df.sort_values(['amfi_code','date'])
nav_df['daily_return'] = nav_df.groupby('amfi_code')['nav'].pct_change()
returns = nav_df.dropna(subset=['daily_return'])

sample_codes  = [100016, 119598, 119120]
n_simulations = 1000
n_days        = 1260   # 5 trading years
np.random.seed(42)

fig, axes = plt.subplots(1, 3, figsize=(16, 6))
fig.suptitle('Monte Carlo NAV Simulation - 5 Year Projection (1,000 paths)', fontsize=13, fontweight='bold')

for ax, code in zip(axes, sample_codes):
    r = returns[returns['amfi_code']==code]['daily_return']
    mu, sigma = r.mean(), r.std()
    nav0 = nav_df[nav_df['amfi_code']==code].sort_values('date').iloc[-1]['nav']
    daily_r = np.random.normal(mu, sigma, (n_days, n_simulations))
    paths   = nav0 * np.cumprod(1 + daily_r, axis=0)
    pct5    = np.percentile(paths, 5, axis=1)
    pct95   = np.percentile(paths, 95, axis=1)
    median  = np.percentile(paths, 50, axis=1)
    for i in range(50):
        ax.plot(paths[:, i], color='#1565C0', alpha=0.04, linewidth=0.5)
    ax.fill_between(range(n_days), pct5, pct95, alpha=0.2, color='#1565C0', label='5-95th pct')
    ax.plot(median, color='#1565C0', linewidth=2, label=f'Median: Rs.{median[-1]:.0f}')
    ax.axhline(nav0, color='gray', linestyle='--', linewidth=1, alpha=0.7, label=f'Now: Rs.{nav0:.0f}')
    ax.set_title(fund_names.get(code, str(code))[:30], fontsize=9, fontweight='bold')
    ax.set_xlabel('Trading Days (5yr = 1,260)', fontsize=8)
    ax.set_ylabel('NAV (Rs.)', fontsize=8)
    ax.legend(fontsize=7)
    ax.grid(axis='y', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.savefig(REPORTS / 'monte_carlo_simulation.png', bbox_inches='tight', dpi=150)
plt.show()
print("Saved: monte_carlo_simulation.png")


## B4 - Markowitz Efficient Frontier
5,000 random portfolio allocations for 5 equity funds. Star marks the max Sharpe portfolio.

In [ ]:
eq_funds = [119551, 100016, 100033, 119598, 120503]
RF_ANNUAL = 0.065

ret_dict = {}
for c in eq_funds:
    r = returns[returns['amfi_code']==c].set_index('date')['daily_return']
    ret_dict[fund_names.get(c, str(c))[:20]] = r
ret_matrix = pd.DataFrame(ret_dict).dropna()

mu_vec  = ret_matrix.mean().values * 252
cov_mat = ret_matrix.cov().values * 252
n_a     = len(eq_funds)
np.random.seed(0)
results = np.zeros((3, 5000))
weights_store = []

for i in range(5000):
    w = np.random.dirichlet(np.ones(n_a))
    p_ret = np.dot(w, mu_vec)
    p_vol = np.sqrt(np.dot(w.T, np.dot(cov_mat, w)))
    results[0, i] = p_vol * 100
    results[1, i] = p_ret * 100
    results[2, i] = (p_ret - RF_ANNUAL) / p_vol
    weights_store.append(w)

max_idx = results[2].argmax()
fig, ax = plt.subplots(figsize=(10, 7))
sc = ax.scatter(results[0], results[1], c=results[2], cmap='RdYlGn', alpha=0.5, s=8)
plt.colorbar(sc, ax=ax, label='Sharpe Ratio')
ax.scatter(results[0, max_idx], results[1, max_idx],
           color='#1565C0', s=200, marker='*', zorder=5,
           label=f'Max Sharpe = {results[2,max_idx]:.2f}')
ax.set_xlabel('Annualised Volatility (%)', fontsize=11)
ax.set_ylabel('Annualised Return (%)', fontsize=11)
ax.set_title('Markowitz Efficient Frontier - 5 Equity Funds', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig(REPORTS / 'efficient_frontier.png', bbox_inches='tight', dpi=150)
plt.show()
print("Saved: efficient_frontier.png")
names_list = [fund_names.get(c, str(c))[:35] for c in eq_funds]
print("Max Sharpe Portfolio:")
for name, w in zip(names_list, weights_store[max_idx]):
    print(f"  {name:<38}: {w*100:.1f}%")
print(f"  Return: {results[1,max_idx]:.2f}%  Vol: {results[0,max_idx]:.2f}%  Sharpe: {results[2,max_idx]:.3f}")


## B5 - Automated HTML Email Report
Generates a weekly performance summary as a self-contained HTML file ready to email.

In [ ]:
from datetime import datetime

scorecard  = pd.read_csv(REPORTS / 'fund_scorecard.csv').head(5)
var_rpt    = pd.read_csv(REPORTS / 'var_cvar_report.csv').head(5)
sip_data   = pd.read_csv(PROC / '04_monthly_sip_inflows_clean.csv')
latest_sip = sip_data.sort_values('month').iloc[-1]
rdate      = datetime.now().strftime("%d %B %Y")

def tbl(df, cols, headers):
    hdr  = "".join(f'<th style="padding:7px 10px;background:#1565C0;color:white">{h}</th>' for h in headers)
    rows = ""
    for _, row in df.iterrows():
        cells = "".join(f'<td style="padding:6px 10px;border-bottom:1px solid #eee">{row[c]}</td>' for c in cols)
        rows += f"<tr>{cells}</tr>"
    return f'<table style="width:100%;border-collapse:collapse;font-size:13px"><thead><tr>{hdr}</tr></thead><tbody>{rows}</tbody></table>'

top5_tbl = tbl(scorecard, ['scheme_name','cagr_3yr_pct','sharpe_ratio','composite_score'],
               ['Fund','3yr CAGR','Sharpe','Score'])
var5_tbl = tbl(var_rpt, ['scheme_name','var_95_pct','cvar_95_pct'], ['Fund','VaR 95%','CVaR 95%'])

kpi_style = 'flex:1;background:#E3F2FD;border-radius:8px;padding:12px;text-align:center'
val_style = 'font-size:18px;font-weight:700;color:#1565C0'
lbl_style = 'font-size:11px;color:#546E7A'

html_lines = [
    '<!DOCTYPE html><html><head><meta charset="UTF-8"></head>',
    '<body style="font-family:Segoe UI,sans-serif;background:#F0F4F8;margin:0;padding:20px">',
    '<div style="max-width:680px;margin:auto;background:white;border-radius:10px;overflow:hidden">',
    '<div style="background:#0D47A1;color:white;padding:24px 30px">',
    f'<h1 style="margin:0;font-size:20px">Bluestock Fintech Weekly MF Report</h1>',
    f'<p style="margin:6px 0 0;opacity:0.8;font-size:12px">Week ending {rdate}</p>',
    '</div>',
    '<div style="padding:24px 30px">',
    f'<div style="display:flex;gap:12px;margin-bottom:20px">',
    f'<div style="{kpi_style}"><div style="{val_style}">Rs.81L Cr</div><div style="{lbl_style}">Industry AUM</div></div>',
    f'<div style="{kpi_style}"><div style="{val_style}">Rs.{int(latest_sip["sip_inflow_crore"]):,} Cr</div><div style="{lbl_style}">SIP Inflow</div></div>',
    f'<div style="{kpi_style}"><div style="{val_style}">26.12 Cr</div><div style="{lbl_style}">Folios</div></div>',
    '</div>',
    '<h2 style="color:#0D47A1;font-size:14px">Top 5 Funds - Composite Scorecard</h2>',
    top5_tbl,
    '<h2 style="color:#C62828;font-size:14px;margin-top:20px">Risk Alert - Highest VaR Funds</h2>',
    var5_tbl,
    '<p style="font-size:11px;color:#78909C;margin-top:16px">Auto-generated by Bluestock MF Analytics. Educational purposes only.</p>',
    '</div>',
    '<div style="background:#F5F5F5;padding:12px 30px;font-size:11px;color:#78909C;text-align:center">',
    'Bluestock Fintech Pvt. Ltd.',
    '</div></div></body></html>',
]
html = "\n".join(html_lines)

out = REPORTS / f'weekly_report_{datetime.now().strftime("%Y%m%d")}.html'
out.write_text(html, encoding='utf-8')
print(f"HTML report saved: {out.name} ({out.stat().st_size/1024:.1f} KB)")
print("Open in browser to preview.")
